Week 9 version  
Perfecting  
GS on fine level  
Then projecting to a coarse level  
Then solving in that coarse level (using the inverse)  
Then projecting back to the fine level  
GS again  
Return solution  
Compare solution to known true answer

In [ ]:
# import necessary packages
from ngsolve import *
from ngsolve.webgui import Draw
import matplotlib.pyplot as plt
import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import norm,inv
import time

In [ ]:
# Set up two levels of meshes
coarse_mesh = Mesh(unit_square.GenerateMesh(maxh=1/16))
fesc = H1(coarse_mesh, order=1, dirichlet="left|right")
fine_mesh = Mesh(coarse_mesh.ngmesh.Copy())
fesf = H1(fine_mesh, order=1, dirichlet="left|right")
fesf.mesh.Refine()
PT = fesf.Prolongation().CreateMatrix(fesf.mesh.levels-1)
P = PT.CreateTranspose()

In [ ]:
# Setup the PDE to solve
u, v = fesf.TnT()
a = BilinearForm(grad(u)*grad(v)*dx).Assemble()
f = LinearForm(fesf).Assemble()
a_fine = a.GetMatrixLevel()
# Initial iterate
x0 = CoefficientFunction(sin(pi*x)*sin(pi*y)+(1/10)*sin(10*pi*x)*sin(10*pi*y))

In [ ]:
# Gauss Seidel Forward Smoothing
m = a_fine.CreateSmoother(fesf.FreeDofs(), GS=True)
xi = GridFunction(fesf)
xi.Set(x0)
yi = GridFunction(fesf)
s = Draw(xi,fine_mesh,settings={"camera": {"transformations": [{"type": "rotateX", "angle": -60}]},"deformation":0.5})
smootha = a_fine
for i in range(20):
    smootha = m @ smootha
    smootha.Mult(xi.vec, yi.vec.data)
    xi.vec.data -= yi.vec
    #time.sleep(1)
    s.Redraw()



In [ ]:
# The coarse space solve
phi, nu = fesc.TnT()
a_coarse = BilinearForm(grad(phi)*grad(nu)*dx).Assemble()
f_coarse = LinearForm(fesc)
f_coarse.vec.data += P * xi.vec
f_coarse.Assemble()
xcoarse = GridFunction(fesc)
xcoarse.vec.data += a_coarse.mat.Inverse(freedofs=fesc.FreeDofs()) * f_coarse.vec
Draw(xcoarse,settings={"camera": {"transformations": [{"type": "rotateX", "angle": -60}]},"deformation":0.5})

In [ ]:
x_fine = GridFunction(fesf)
x_fine.Set(0, BND)
x_fine.vec.data += PT * xcoarse.vec

Draw(x_fine,fine_mesh)

In [ ]:
# GS backward smoothing
m2 = a_fine.CreateSmoother(fesf.FreeDofs(), GS=True)
yfine = GridFunction(fesf)
xi.vec.data = x_fine.vec.Copy()
s2 = Draw(xi,fine_mesh,settings={"camera": {"transformations": [{"type": "rotateX", "angle": -60}]},"deformation":0.5})
smootha = a_fine
for i in range(20):
    smootha = m2 @ smootha
    smootha.Mult(x_fine.vec, yfine.vec.data)
    xi.vec.data -= yi.vec
    #time.sleep(1)
    s2.Redraw()

In [ ]:
# Report result
# L2Norm
l2n = xi.vec.Norm()
# Energy norm
enorm = sqrt(InnerProduct(xi.vec, a_fine * xi.vec))
print(f'Convergence after 1 two-level solve ||x||=', l2n, ', and ||x||_A=', enorm)

Below, we have a function used to check if the A matrices at both levels are spd. They are.

In [ ]:
# Testing if a matrix A is SPD via z^T A z > 0 with z = random vectors
def spd_check(A,fes,max_iter=10000):
    iter = 0
    w = 1
    while(w > 0 and iter < max_iter):
        z = GridFunction(fes)
        v = z.vec.CreateVector()
        v.SetRandom()
        z.vec.data = Projector(fes.FreeDofs(), True) * v
        w = GridFunction(fes)
        w = InnerProduct(z.vec, A * z.vec)
        if(w == 0):
            print('Matrix is not SPD')
            return False
        iter += 1
        #print(w)
    return True

fine_spd = spd_check(a_fine, fesf)
coarse_spd = spd_check(a_coarse.mat, fesc)
fine_spd, coarse_spd


In [ ]:
help(xi.vec)